In [ ]:

import time
import psutil
import threading
from pathlib import Path
import numpy as np
import cv2
import os
import tracemalloc
import csv
from PIL import Image
from openflexure_microscope_client import MicroscopeClient

# ============================
# CPU MONITOR SETUP
# ============================
cpu_usage_list = []
monitoring = True

def monitor_cpu():
    while monitoring:
        cpu = psutil.cpu_percent(interval=0.2)
        cpu_usage_list.append(cpu)

cpu_thread = threading.Thread(target=monitor_cpu, daemon=True)
cpu_thread.start()

# ============================
# Connect to microscope
# ============================
microscope = MicroscopeClient("10.150.79.160")

# ============================
# Parameters
# ============================
coarse_step_size = 200
coarse_num_steps = 3
fine_step_size = 50
fine_num_steps = 2
settle_time = 0.01
patience = 2

# ============================
# Helper Functions
# ============================
def compute_laplacian_variance(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def move_microscope_rel(z_delta):
    microscope.move_rel({"x": 0, "y": 0, "z": z_delta})
    time.sleep(settle_time)

def move_microscope_abs(z_target):
    microscope.move({
        "x": microscope.position['x'],
        "y": microscope.position['y'],
        "z": z_target
    })
    time.sleep(settle_time)

def capture_image():
    return np.array(microscope.grab_image())

# ============================
# Direction Evaluation
# ============================
def evaluate_focus_direction(step_size, num_steps):
    base_z = microscope.position['z']
    variances = []

    # Move upward
    for _ in range(num_steps):
        move_microscope_rel(step_size)
        image = capture_image()
        var = compute_laplacian_variance(image)
        variances.append((var, microscope.position['z']))

    # Return to base
    move_microscope_rel(-step_size * num_steps)

    # Move downward
    for _ in range(num_steps):
        move_microscope_rel(-step_size)
        image = capture_image()
        var = compute_laplacian_variance(image)
        variances.append((var, microscope.position['z']))

    # Return to base
    move_microscope_rel(step_size * num_steps)

    best_var, best_z = max(variances, key=lambda x: x[0])
    direction = 1 if best_z > base_z else -1
    return direction

# ============================
# Hill Climb Focus
# ============================
def move_until_focus_drops(step_size, direction, patience=3):
    best_var = -1
    best_z = microscope.position['z']
    stagnant_steps = 0

    while stagnant_steps < patience:
        move_microscope_rel(direction * step_size)
        image = capture_image()
        var = compute_laplacian_variance(image)

        if var > best_var + 1e-4:
            best_var = var
            best_z = microscope.position['z']
            stagnant_steps = 0
        else:
            stagnant_steps += 1

    # Return to best focus
    move_microscope_abs(best_z)
    return best_z, best_var

# ============================
# Coarse Focus
# ============================
def coarse_focus():
    print("Evaluating direction for coarse focus...")
    direction = evaluate_focus_direction(coarse_step_size, coarse_num_steps)
    print(f"Coarse focus direction: {'up' if direction == 1 else 'down'}")

    best_z, best_var = move_until_focus_drops(coarse_step_size, direction, patience)
    print(f"Coarse focus at Z={best_z} with variance={best_var:.2f}")
    return best_z

# ============================
# Fine Focus
# ============================
def fine_focus(coarse_z):
    print("Starting fine focus...")
    move_microscope_abs(coarse_z)
    base_z = coarse_z
    variances = []

    for i in range(-fine_num_steps, fine_num_steps + 1):
        target_z = base_z + i * fine_step_size
        move_microscope_abs(target_z)
        image = capture_image()
        var = compute_laplacian_variance(image)
        variances.append((var, target_z))

    best_var, best_z = max(variances, key=lambda x: x[0])
    move_microscope_abs(best_z)
    print(f"Fine focus done at Z={best_z} with variance={best_var:.2f}")
    return best_z, best_var

# ============================
# Autofocus Pipeline
# ============================
def autofocus():
    coarse_z = coarse_focus()
    move_microscope_abs(coarse_z)
    fine_z, fine_var = fine_focus(coarse_z)
    move_microscope_abs(fine_z)
    return fine_z, fine_var

# ============================
# Output Directory
# ============================
desktop = Path.home() / "Desktop"
output_dir = desktop / "microscope 28.7.1cancer."
output_dir.mkdir(parents=True, exist_ok=True)

# ============================
# Store Initial Position
# ============================
starting_pos = microscope.position.copy()

# ============================
# START RAM TRACKING
# ============================
process = psutil.Process(os.getpid())
os_ram_start = process.memory_info().rss
tracemalloc.start()

# ============================
# TOTAL RUN START
# ============================
total_start = time.time()

# ============================
# Capture Initial Image
# ============================
start_time = time.time()
init_z = microscope.position['z']
initial_image = capture_image()
initial_var = compute_laplacian_variance(initial_image)

# Save ONLY initial image
initial_filename = output_dir / "initial_image2.png"
Image.fromarray(initial_image).save(initial_filename)

print(f"\n📍 Initial Z={init_z}")
print(f"🔍 Initial Variance={initial_var:.2f}")
print(f"💾 Initial image saved at {initial_filename}")

# ============================
# Autofocus
# ============================
best_z, best_var = autofocus()

# ============================
# Capture Final Image ONLY
# ============================
final_image = capture_image()
final_filename = output_dir / "final_focused_image20.png"
Image.fromarray(final_image).save(final_filename)

# ============================
# End Timing
# ============================
total_end = time.time()
total_elapsed = total_end - total_start

print(f"\n📏 Final Z={best_z}")
print(f"🔍 Final Variance={best_var:.2f}")
print(f"💾 Final image saved at {final_filename}")

# ============================
# STOP RAM TRACKING
# ============================
current_py_mem, peak_py_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()
os_ram_end = process.memory_info().rss
os_ram_diff = os_ram_end - os_ram_start

# ============================
# Return to Starting Position
# ============================
starting_pos['z'] = best_z
microscope.move(starting_pos)

# ============================
# Stop CPU Monitoring
# ============================
monitoring = False
cpu_thread.join()

# Safety fallback
if len(cpu_usage_list) == 0:
    cpu_usage_list.append(psutil.cpu_percent(interval=0.1))

mean_cpu_usage = np.mean(cpu_usage_list)
std_cpu_usage = np.std(cpu_usage_list)
max_cpu_usage = np.max(cpu_usage_list)

# ============================
# Final Summary
# ============================
print("\n==============================")
print("📊 PERFORMANCE SUMMARY")
print("==============================")

print(f"\n⏱ Total Run Time: {total_elapsed:.2f} seconds")

print("\n💻 CPU Usage Summary")
print(f"   • Mean CPU usage: {mean_cpu_usage:.2f}%")
print(f"   • Std CPU usage:  {std_cpu_usage:.2f}%")
print(f"   • Max CPU usage:  {max_cpu_usage:.2f}%")

print("\n🧠 Memory Usage Summary")
print(f"   • Peak Python memory:   {peak_py_mem / (1024 * 1024):.2f} MB")
print(f"   • Net OS RAM footprint: {os_ram_diff / (1024 * 1024):.2f} MB")

print(f"\n📂 Images saved in '{output_dir}'")
print("✅ Autofocus routine complete.")

# ============================
# Save Results to CSV
# ============================
csv_file = output_dir / "results.csv"
file_exists = csv_file.exists()

peak_py_mem_mb = peak_py_mem / (1024 * 1024)
os_ram_diff_mb = os_ram_diff / (1024 * 1024)
variance_change = best_var - initial_var

with open(csv_file, mode='a', newline='') as file:
    writer = csv.writer(file)
    
    # Write the header only if the file is being created for the first time
    if not file_exists:
        writer.writerow([
            "Z_Initial", "Z_Final", 
            "Initial_Variance", "Final_Variance", "Variance_Change",
            "Total_Execution_Time_s", "Mean_CPU_Usage_percent", "Max_CPU_Usage_percent",
            "Peak_Py_Mem_MB", "Net_OS_RAM_MB"
        ])
    
    # Write the data for this specific run
    writer.writerow([
        init_z, best_z, 
        f"{initial_var:.4f}", f"{best_var:.4f}", f"{variance_change:.4f}",
        f"{total_elapsed:.2f}", f"{mean_cpu_usage:.2f}", f"{max_cpu_usage:.2f}",
        f"{peak_py_mem_mb:.2f}", f"{os_ram_diff_mb:.2f}"
    ])

print(f"📁 Results saved successfully to {csv_file}")

